In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Peptipedia2.0)

This notebook curates the **Peptipedia2.0** dataset from a pivoted table where each peptide sequence is annotated with multiple binary flags describing toxicity effects. The pipeline extracts task-specific subsets (e.g., hemolytic, cytotoxic, neurotoxic, toxic), performs duplicate consistency checks for toxicity tasks, and exports standardized CSV files plus metadata.

- **Toxic effect / endpoint:** hemolytic, cytotoxic, neurotoxic, toxic
- **Source:** Peptipedia2.0
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the pivoted annotation table** (`pivoted_sequences.csv`) containing:
  - a `sequence` column,
  - multiple binary indicator columns such as `Toxic`, `Hemolytic`, `Cytotoxic`, `Neurotoxin`
- **Splits the dataset by toxicity effect** (positive-only subsets):
  - `toxic`, `hemolytic`, `cytotoxic`, `neurotoxic` are extracted by filtering rows where the corresponding flag equals 1,
  - each subset is standardized to `["sequence", "label"]` with `label = 1`.
- **Checks duplicated sequences** for the toxicity subsets:
  - identical sequences are collapsed when consistent,
  - conflicting cases are reported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - Toxicity datasets:
    - `processed_hemolytic_dataset.csv`
    - `processed_toxic_dataset.csv`
    - `processed_cytotoxic_dataset.csv`
    - `processed_neurotoxic_dataset.csv`
    - `detected_error_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "Peptipedia2.0"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_peptipedia = pd.read_csv(f"{PATH_INPUT}/{name_source}/pivoted_sequences.csv")

- Split the dataset by toxic effect

In [4]:
df_toxic = (
    df_peptipedia
    .loc[df_peptipedia["Toxic"] == 1, ["sequence", "Toxic"]]
    .rename(columns={"Toxic": "label"})
)

In [5]:
df_hemolytic = (
    df_peptipedia
    .loc[df_peptipedia["Hemolytic"] == 1, ["sequence", "Hemolytic"]]
    .rename(columns={"Hemolytic": "label"})
)

In [6]:
df_cytotoxic = (
    df_peptipedia
    .loc[df_peptipedia["Cytotoxic"] == 1, ["sequence", "Cytotoxic"]]
    .rename(columns={"Cytotoxic": "label"})
)

In [7]:
df_neurotoxic = (
    df_peptipedia
    .loc[df_peptipedia["Neurotoxin"] == 1, ["sequence", "Neurotoxin"]]
    .rename(columns={"Neurotoxin": "label"})
)

- Checking duplicates

In [8]:
df_remove_duplicated_toxic, df_errors_toxic, df_unique_toxic = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")

In [9]:
df_remove_duplicated_hemo, df_errors_hemo, df_unique_hemo = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_cyto, df_errors_cyto, df_unique_cyto = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [11]:
df_remove_duplicated_neuro, df_errors_neuro, df_unique_neuro = processing_duplicated(df_neurotoxic, group_seq="sequence", sort_key="label")

In [12]:
df_full_toxic = pd.concat([df_unique_toxic, df_remove_duplicated_toxic], axis=0)
df_full_toxic.shape

(19104, 2)

In [13]:
df_full_hemo = pd.concat([df_unique_hemo, df_remove_duplicated_hemo], axis=0)
df_full_hemo.shape

(1304, 2)

In [14]:
df_full_cyto = pd.concat([df_unique_cyto, df_remove_duplicated_cyto], axis=0)
df_full_cyto.shape

(1470, 2)

In [15]:
df_full_neuro = pd.concat([df_unique_neuro, df_remove_duplicated_neuro], axis=0)
df_full_neuro.shape

(577, 2)

In [16]:
df_full = pd.concat([df_full_toxic, df_full_cyto,
                     df_full_hemo, df_full_neuro],
                     ignore_index=True)

In [17]:
df_errors = pd.concat([df_errors_toxic, df_errors_cyto,
                     df_errors_hemo, df_errors_neuro],
                     ignore_index=True)
df_errors.shape

(0, 1)

- Working with metada

In [18]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [19]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_peptipedia)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Database',
 'static-dynamic': 'Dynamic',
 'license': 'DbCL',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 3, 23, 0, 0),
 'download date': Timestamp('2026-01-08 00:00:00'),
 'file format': 'csv',
 'peptide property': 'anti HIV, anti nematode, anti coronaviridae, enzyme inhibitor, anti influenza virus, chemotactic, anti herpes simplex virus, anti poxviridae, antidiabetic, antitoxin peptides, antimollicute, protein-binding, opioid, antiviral, insecticidal, taste, immunostimulating, anti gram positive, protease inhibitor, anti andes virus, antiendotoxin, anti west nile virus, anti newcastle disease virus, anti ebola virus, anti human metapneumovirus, anti varicella zoster virus, anti cowpox virus, anti arenaviridae, anti human t-cell leukaemia virus 1, metabolic, anti murine norovirus, anti foot and mouth disease virus, anti avian myeloblastosis virus, anti tacaribe virus, transit, anti adenoviridae, anti papillomaviridae, anti feline coronavi

- Exporting data

In [20]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [21]:
df_full_hemo.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)
df_full_cyto.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_neuro.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_neurotoxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)